In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

c:\Users\ELITEBOOK\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Chargement des données

# Vectorisation TF-IDF

Prépare les features texte pour l'approche A (ML classique, voir README). Deux vectorizers séparés — un pour `intitule`, un pour `description` — car leurs vocabulaires ont des tailles très différentes et n'ont pas besoin des mêmes réglages.

`min_df`/`max_features` sont choisis par validation croisée sur `train` uniquement : `test` reste intact jusqu'à l'évaluation finale.

In [2]:
df = pd.read_csv(r"..\data\interim\train.csv")
test_df = pd.read_csv(r"..\data\interim\test.csv")
print(f"train: {len(df)} lignes, test: {len(test_df)} lignes")
df.head(3)

train: 80 lignes, test: 20 lignes


,offer_id,intitule,description,seniority,contract_type,home_office_policy,min_salary,max_salary,salary_period,skills
0,6798640,Lead Développeur Fullstack H/F,"Acteur majeur de la tech en Europe, notre clie...",lead,cdi,hybrid,NaN,NaN,NaN,"java, spring, angular"
1,6781955,Développeur/euse Expérimenté/e - Java & Angula...,"Description de l'entrepriseSopra Steria, acteu...",confirme,cdi,on_site,NaN,NaN,NaN,"java, spring boot, angular, docker, kubernetes..."
2,6778039,Développeur JAVA (H/F),Description du poste : En tant qu'organisateur...,confirme,cdi,on_site,NaN,NaN,NaN,"java, spring, react, agile"


## 2. Le problème de dimensionnalité

Avec seulement 80 lignes en `train`, un vocabulaire sans limite explose largement au-delà du nombre de lignes — surtout pour `description` (texte long). C'est ce qui motive `min_df`/`max_features` ci-dessous, pas juste une précaution théorique.

In [3]:
for col in ["intitule", "description"]:
    vocab_size = TfidfVectorizer().fit(df[col]).get_feature_names_out().shape[0]
    print(f"{col}: {vocab_size} mots dans le vocabulaire sans limite (vs {len(df)} lignes en train)")

intitule: 127 mots dans le vocabulaire sans limite (vs 80 lignes en train)
description: 3831 mots dans le vocabulaire sans limite (vs 80 lignes en train)


## 3. Choisir `max_features` par validation croisée (sur `train` uniquement)

Pas de split `val` séparé — trop peu de données pour se permettre d'en sacrifier un (voir discussion). On utilise un `StratifiedKFold` sur `train`, avec `seniority` comme cible de test (4 classes, au moins 6 exemples chacune — sûr pour la validation croisée, contrairement à `contract_type` qui n'a qu'1 exemple `alternance`).

Ce choix de `max_features` sert pour `description` uniquement — `intitule` a un vocabulaire naturellement petit (voir section 2), pas besoin de le plafonner aussi agressivement.

**Découverte en cours de route :** sans `class_weight="balanced"`, toutes les valeurs de `max_features` donnaient exactement le même F1 macro (0.218) — le modèle prédisait toujours la classe majoritaire (`confirme`, 62/80) sans jamais regarder le texte, à cause du déséquilibre des classes. `class_weight="balanced"` force le modèle à réellement essayer d'apprendre les classes minoritaires, ce qui est nécessaire avant même de pouvoir juger l'effet de `max_features`.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y = df["seniority"]

candidate_max_features = [50, 100, 150, 200, 300, None]
cv_results = {}

for max_features in candidate_max_features:
    pipeline = Pipeline(
        [
            ("tfidf", TfidfVectorizer(min_df=2, max_features=max_features)),
            ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    )
    scores = cross_val_score(pipeline, df["description"], y, cv=cv, scoring="f1_macro")
    cv_results[max_features] = (scores.mean(), scores.std())
    label = max_features if max_features is not None else "aucune limite"
    print(f"max_features={label}: F1 macro = {scores.mean():.3f} (+/- {scores.std():.3f})")

max_features=50: F1 macro = 0.317 (+/- 0.063)
max_features=100: F1 macro = 0.413 (+/- 0.054)
max_features=150: F1 macro = 0.351 (+/- 0.031)
max_features=200: F1 macro = 0.356 (+/- 0.032)
max_features=300: F1 macro = 0.386 (+/- 0.038)
max_features=aucune limite: F1 macro = 0.417 (+/- 0.049)


**Règle du "1 écart-type" plutôt que l'argmax brut :** avec seulement 80 lignes, les scores se chevauchent dans le bruit — l'argmax choisirait `None` (aucune limite, 1928 colonnes après `min_df=2`), ce qui recrée exactement le problème de dimension qu'on veut éviter. On garde plutôt la valeur la **plus petite** dont le score reste à moins d'un écart-type du meilleur.

In [6]:
best_mean, best_std = max(cv_results.values(), key=lambda ms: ms[0])
threshold = best_mean - best_std

eligible = [mf for mf, (mean, _) in cv_results.items() if mean >= threshold]
best_max_features = min(eligible, key=lambda mf: mf if mf is not None else float("inf"))

print(f"Meilleur score brut : {best_mean:.3f} +/- {best_std:.3f} (seuil retenu : {threshold:.3f})")
print(f"max_features retenu (regle du 1 ecart-type, le plus simple parmi les eligibles) : {best_max_features}")

Meilleur score brut : 0.417 +/- 0.049 (seuil retenu : 0.368)
max_features retenu (regle du 1 ecart-type, le plus simple parmi les eligibles) : 100


## 4. Vectorisation finale

Un vectorizer par colonne, ajusté (`fit`) sur `train` uniquement, puis appliqué (`transform`, jamais `fit`) sur `test` — un mot inédit dans `test` sera ignoré (voir discussion sur le vocabulaire fermé).

In [7]:
vectorizer_intitule = TfidfVectorizer(min_df=2)
vectorizer_description = TfidfVectorizer(min_df=2, max_features=best_max_features)

X_train_intitule = vectorizer_intitule.fit_transform(df["intitule"])
X_train_description = vectorizer_description.fit_transform(df["description"])

X_test_intitule = vectorizer_intitule.transform(test_df["intitule"])
X_test_description = vectorizer_description.transform(test_df["description"])

print("intitule    - train:", X_train_intitule.shape, "test:", X_test_intitule.shape)
print("description - train:", X_train_description.shape, "test:", X_test_description.shape)

intitule    - train: (80, 46) test: (20, 46)
description - train: (80, 100) test: (20, 100)


## Conclusion

Deux découvertes en cours de route, plus importantes que le réglage `max_features` lui-même :

1. **Le déséquilibre des classes masquait tout** : sans `class_weight="balanced"`, le F1 macro était identique (0.218) pour n'importe quelle valeur de `max_features` — le modèle prédisait toujours `confirme` (62/80) sans jamais regarder le texte. Corrigé, les scores varient enfin selon `max_features` (0.317 à 0.417).
2. **L'argmax brut aurait été trompeur** : la meilleure moyenne CV correspond à "aucune limite" (1928 colonnes pour 80 lignes — exactement le problème qu'on voulait éviter), mais l'écart avec `max_features=100` est dans le bruit (barres d'erreur qui se chevauchent). La règle du 1 écart-type retient `100`, bien plus raisonnable.

**Résultat final :**
- `intitule` : `min_df=2` seul, 46 colonnes (vocabulaire naturellement petit)
- `description` : `min_df=2` + `max_features=100`, 100 colonnes
- Les deux vectorizers sont ajustés (`fit`) sur `train` uniquement, puis appliqués (`transform`) sur `test` — un mot inédit dans `test` est silencieusement ignoré (limite du "vocabulaire fermé" de cette approche, pas une erreur)
- Prochaine étape (Milestone 2) : entraîner les classifieurs par champ (`seniority`, `contract_type`, `home_office_policy`) avec `class_weight="balanced"` sur ces features — le déséquilibre des classes concerne tout le dataset, pas juste cette étape de vectorisation.